# Data Validation Notebook

This notebook validates data quality and returns a status for workflow decisions.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Get input parameters
dbutils.widgets.text("data_size", "100", "Number of data points")
dbutils.widgets.text("seed", "42", "Random seed")

data_size = int(dbutils.widgets.get("data_size"))
seed = int(dbutils.widgets.get("seed"))

print(f"Validating data with size={data_size}, seed={seed}")

In [ ]:
# Generate test data
np.random.seed(seed)
data = {
    'x': np.random.randn(data_size),
    'y': np.random.randn(data_size) * 2 + 5,
    'category': np.random.choice(['A', 'B', 'C'], data_size)
}
df = pd.DataFrame(data)

print(f"Generated {len(df)} data points")
print(df.head())

In [ ]:
# Share the actual data via task_values (for other notebooks to use)
dbutils.jobs.taskValues.set("raw_data_mean_x", df['x'].mean())
dbutils.jobs.taskValues.set("raw_data_mean_y", df['y'].mean())
dbutils.jobs.taskValues.set("raw_data_std_x", df['x'].std())
dbutils.jobs.taskValues.set("raw_data_std_y", df['y'].std())
dbutils.jobs.taskValues.set("data_size_actual", len(df))
dbutils.jobs.taskValues.set("category_distribution", df['category'].value_counts().to_dict())

print("✅ Shared data statistics via task_values")

In [ ]:
# Perform data quality validation
validation_results = {
    'has_nulls': df.isnull().any().any(),
    'size_check': len(df) == data_size,
    'x_range_valid': df['x'].min() > -5 and df['x'].max() < 5,
    'y_range_valid': df['y'].min() > -5 and df['y'].max() < 15,
    'categories_valid': set(df['category'].unique()) == {'A', 'B', 'C'}
}

print("Data Quality Checks:")
for check, result in validation_results.items():
    status = "✅" if result else "❌"
    print(f"  {check}: {status}")

In [ ]:
# Determine validation status for workflow decision
all_checks_passed = all([
    not validation_results['has_nulls'],
    validation_results['size_check'],
    validation_results['x_range_valid'],
    validation_results['y_range_valid'],
    validation_results['categories_valid']
])

if all_checks_passed:
    status = "PASSED"
    next_step = "advanced_analysis"
    print("🎉 All data quality checks PASSED!")
else:
    status = "FAILED" 
    next_step = "basic_analysis"
    print("⚠️  Some data quality checks FAILED")

print(f"Recommended next step: {next_step}")

In [ ]:
# Return status for workflow conditional logic
workflow_decision = {
    "status": status,                    # PASSED or FAILED
    "next_step": next_step,             # advanced_analysis or basic_analysis  
}

print(f"🎯 Workflow Decision: {workflow_decision}")

# This exit_value will be used by workflow for conditional logic
dbutils.notebook.exit(workflow_decision)